https://github.com/UKPLab/sentence-transformers/blob/68dfbe643d51f1890e410b6783ca5343620db4fc/sentence_transformers/trainer.py#L620

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, DatasetDict

# Load a dataset (for example, IMDb for sentiment analysis)
imdb_dataset = load_dataset("imdb")
imdb_train_dataset = imdb_dataset['train'].shuffle().select(range(1000))  # Small subset for quick training
imdb_eval_dataset = imdb_dataset['test'].shuffle().select(range(500))

ag_news_dataset = load_dataset("ag_news")
ag_news_train_dataset = ag_news_dataset['train'].shuffle().select(range(1000))  # Small subset for quick training
ag_news_eval_dataset = ag_news_dataset['test'].shuffle().select(range(500))

# Load a pre-trained model and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize the data
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True)

imdb_train_dataset = imdb_train_dataset.map(tokenize, batched=True)
imdb_eval_dataset = imdb_eval_dataset.map(tokenize, batched=True)

ag_news_train_dataset = ag_news_train_dataset.map(tokenize, batched=True)
ag_news_eval_dataset = ag_news_eval_dataset.map(tokenize, batched=True)

'''
train_dataset = {
    "imdb_train_dataset": imdb_train_dataset,
    "ag_news_train_dataset": ag_news_train_dataset,
}

eval_dataset = {
    "imdb_eval_datasett": imdb_eval_dataset,
    "ag_news_eval_dataset": ag_news_eval_dataset,
}
'''

# Create a DatasetDict for training and evaluation
train_dataset = DatasetDict({
    "imdb": imdb_train_dataset,
    "ag_news": ag_news_train_dataset,
})

eval_dataset = DatasetDict({
    "imdb": imdb_eval_dataset,
    "ag_news": ag_news_eval_dataset,
})

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to = 'tensorboard'
)

#print(training_args.report_to)
#training_args.report_to = None

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print('Ready for training!')
# Train the model
trainer.train()

/Users/sergiopaniegoblanco/Documents/Projects/transformers/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sergiopaniegoblanco/Documents/Projects/transformers/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some 